# Regression inference — searchlight steps 15.6–15.10

Continue the first regression Colab using its **completed group ZIPs** (`result_regression_group_<target>_<species>.zip`). Participant ZIPs and run checkpoints are not sufficient: finish **15.3 and 15.5** first. Defaults: EmoB, humans, visual_3; change `SPECIE` to `'D'` for dogs.

The packaged `rsa_utils.py` supplies the exact pipeline inference and atlas-report functions. A float64 GPU streaming reducer computes the null moments for 15.6. NIfTI I/O, z-map writing, connected components, correction and reports run on CPU. One target is staged locally at a time, with progress messages and one final checkpoint ZIP per target.

**Statistics:** population null std (`ddof=0`); z=0 where std ≤ 1e-8; strict positive-tail `z > Z_THRESHOLD`; 26-neighbour clusters; the same empirical maximum-cluster rule as `searchlight.py`. Reports use AAL3 for humans and Czeibert for dogs.

In [ ]:
!pip -q install nibabel pandas scipy pyyaml nilearn ipywidgets matplotlib
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE = '/content/drive/MyDrive/rsa_colab'
DATASET = 'EmoB'
SPECIE = 'H'                 # 'H' or 'D'
MODEL = 'basic-block'
REGRESSION_MODEL = 'visual_3'
MASK_TYPE = 'b_GreyMatter2mmB'
RADIUS = None                # H=4, D=3; set explicitly if different upstream
DIS_METHOD = 'correlation'
RESULTS_DIR = f'{BASE}/results_regression_EmoB'
OUT_DIR = f'{BASE}/results_regression_inference_EmoB'
SUPPORT_ZIP = f'{BASE}/regression_inference_support_{DATASET}.zip'
MODELS = None               # all completed group ZIPs; or ['anger', ...]
REPS_GROUP = 1000           # must match the upstream group draws to consume
Z_THRESHOLD = 3.1
CLUSTER_THRESHOLD = 0.05
MIN_DIST_MM = 8.0
DEVICE = 'cuda'             # 'cpu' also supported
WRITE_PERMUTATION_Z = True  # exact step-15.7 outputs; False saves Drive space after 15.8 consumes them
FORCE = False
WORK_ROOT = '/content/regression_inference_work'

In [ ]:
# Load an isolated snapshot, leaving the first notebook's package untouched.
from pathlib import Path, PurePosixPath
import sys, json, zipfile, importlib, hashlib
SUPPORT_ROOT = Path('/content/regression_inference_support')
SUPPORT_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(SUPPORT_ZIP) as zf:
    for info in zf.infolist():
        name = info.filename
        if '\\' in name or ':' in name or name.startswith('/') or '..' in PurePosixPath(name).parts:
            raise ValueError(f'Unsafe archive member: {name}')
        if not info.is_dir():
            destination = SUPPORT_ROOT / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(zf.read(info))
config = json.loads((SUPPORT_ROOT / 'inference_manifest.json').read_text())
for name, expected_hash in config['file_sha256'].items():
    if hashlib.sha256((SUPPORT_ROOT / name).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f'Support file checksum mismatch: {name}')
sys.path.insert(0, str(SUPPORT_ROOT / 'code'))
for name in ('rsa_utils', 'utils', 'preprocess_functions', 'gpu_rsa', 'gpu_regression',
             'run_colab_regression', 'run_colab_regression_inference'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import run_colab_regression_inference as inference
import torch
print('Inference runtime:', inference.VERSION)
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime or set DEVICE="cpu".')
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Preflight: require real mean and all requested permutation means per target.
if config['dataset'] != DATASET or SPECIE not in config['species']:
    raise ValueError('Support package dataset/species does not match settings.')
radius = RADIUS if RADIUS is not None else (4 if SPECIE == 'H' else 3)
ready = inference.discover_groups(RESULTS_DIR, DATASET, MODEL, REGRESSION_MODEL,
    SPECIE, MASK_TYPE, radius, DIS_METHOD, REPS_GROUP, MODELS)
print(f'{len(ready)} completed target group ZIPs are ready.')
print('Outputs:', OUT_DIR)

In [ ]:
written = inference.run_inference(
    RESULTS_DIR, OUT_DIR, SUPPORT_ROOT, dataset=DATASET, model=MODEL,
    regression_model=REGRESSION_MODEL, specie=SPECIE, models=MODELS,
    radius=RADIUS, mask_type=MASK_TYPE, dis_method=DIS_METHOD,
    reps_group=REPS_GROUP, z_threshold=Z_THRESHOLD,
    cluster_threshold=CLUSTER_THRESHOLD, min_dist_mm=MIN_DIST_MM,
    device=DEVICE, work_root=WORK_ROOT, force=FORCE,
    write_permutation_z=WRITE_PERMUTATION_Z)
print('All available targets complete. New result ZIPs:')
for path in written:
    print(path)

## Outputs and rerunning

Each result ZIP contains the new pipeline files under `RSA_regression` and `RSA_regression_rnd`: null mean/std and receipts (**15.6**), real/permutation z-maps (**15.7**), cluster null `.npy`/JSON (**15.8**), corrected map/JSON (**15.9**) and atlas-labelled CSV (**15.10**, including a valid header-only report if no clusters survive). Original group means are not duplicated. If `WRITE_PERMUTATION_Z=False`, the permutation z-maps are still computed and consumed locally but omitted from the export.

Reruns skip compatible completed target ZIPs. Interrupted targets recompute; earlier completed targets remain on Drive. Run the preflight and computation cells again when more group ZIPs arrive. Set `MODELS` explicitly if you want missing targets to be reported as errors. Keep threshold settings consistent with the desired analysis; changing them creates a different result ZIP. This notebook always runs all five steps in order, with `'15.10'` stored as a string to avoid its conversion to 15.1.

To merge results on the workstation, use the updated unpacker (it now accepts CSV reports):
```powershell
& 'C:\ProgramData\anaconda3\python.exe' tools\unpack_results.py 'G:\My Drive\rsa_colab\results_regression_inference_EmoB'
```
Add `--replace` to replace earlier results. Receipts retain Colab source paths for provenance; if rerunning individual downstream steps on the workstation, start again at 15.6 to regenerate local receipts.